# Step 10 — Bosserhof Potentials & Zone Redistribution

**Input:**
- `data/output/09_llm_predictions_merged.gpkg`
- `data/input/zone_targets.gpkg` (layer: `regionbsstructuredata_zone`)

**Output:**
- `data/output/10_final_potentials.gpkg`
- `data/output/10_building_level_redistributed.gpkg`
- `data/output/10_redistribution_validation.csv`
- `data/output/10_redistribution_allocation_log.csv`

**Zone column mapping:** `ZONE_COLUMN_MAP` in `config.py` translates raw VISUM column names (e.g. `SG_3_BE~17`) to the 7 pipeline activity names (`Workers`, `School`, …). Change this dict to adapt to a different zone data source.

In [ ]:
import sys
sys.path.insert(0, str(__import__('pathlib').Path('..').resolve()))
from config import *

import geopandas as gpd
import pandas as pd
import numpy as np
import ast
import warnings
warnings.filterwarnings('ignore')
print('Config loaded')

## 1. Load merged LLM predictions

In [ ]:
df = gpd.read_file(LLM_MERGED_FILE)
print(f'Loaded {len(df):,} classified buildings')
df = df.dropna(subset=['bosserhof_class']).copy()
df = df[df['volume_m3'] >= MIN_CONDENSED_VOLUME_M3].copy()
print(f'After filters: {len(df):,}')

## 2. Normalize Bosserhof class names and compute potentials

In [ ]:
df['bosserhof_class_norm'] = (
    df['bosserhof_class'].astype(str).str.strip().str.lower()
    .replace(['none', 'nan'], np.nan)
    .map(lambda x: BOSSERHOF_NORMALIZATION_MAP.get(x, x) if pd.notna(x) else x)
)

weight_dict = {k.lower(): v for k, v in BOSSERHOF_WEIGHTS.items()}
df['potentials_raw'] = df['volume_m3'] * df['bosserhof_class_norm'].map(weight_dict)

unmapped = df[df['potentials_raw'].isna()]['bosserhof_class_norm'].dropna().unique()
if len(unmapped) > 0:
    print(f'WARNING: {len(unmapped)} bosserhof classes have no weight — add to BOSSERHOF_WEIGHTS in config.py:')
    for cls in sorted(unmapped): print(f'  {cls}')

def apply_caps(df_in):
    df_out = df_in.copy()
    for cls in STRICT_CAP_CLASSES:
        mask = df_out['bosserhof_class_norm'] == cls
        if mask.any():
            cap = df_out.loc[mask, 'volume_m3'].quantile(0.75)
            df_out.loc[mask, 'potentials_raw'] = df_out.loc[mask, 'volume_m3'].clip(upper=cap) * weight_dict.get(cls, 0)
    for cls in LARGE_FORMAT_CAP_CLASSES:
        mask = df_out['bosserhof_class_norm'] == cls
        if mask.any():
            cap = df_out.loc[mask, 'volume_m3'].quantile(0.95)
            df_out.loc[mask, 'potentials_raw'] = df_out.loc[mask, 'volume_m3'].clip(upper=cap) * weight_dict.get(cls, 0)
    return df_out

df = apply_caps(df)
df['potentials'] = df['potentials_raw'].round(2)
df_final = df.dropna(subset=['potentials']).copy()
print(f'Buildings with valid potentials: {len(df_final):,}')
df_final.to_file(POTENTIALS_FILE, driver='GPKG')
print(f'Saved → {POTENTIALS_FILE}')

## 3. Load zone targets and apply column mapping

The zone file uses raw VISUM column names. `ZONE_COLUMN_MAP` in `config.py` maps them to the
7 pipeline activity names. A list value means those source columns are summed.

In [ ]:
zones = gpd.read_file(ZONE_TARGETS_FILE, layer=ZONE_LAYER)
zones = zones.to_crs(TARGET_CRS)
print(f'Loaded {len(zones):,} zones from layer "{ZONE_LAYER}"')

# Apply column mapping (VISUM SG_ names → pipeline activity names)
if ZONE_COLUMN_MAP:
    for target_col, source in ZONE_COLUMN_MAP.items():
        if target_col in zones.columns:
            continue   # already has the friendly name — skip
        if isinstance(source, list):
            available = [c for c in source if c in zones.columns]
            if not available:
                print(f'WARNING: none of {source} found for "{target_col}" — column will be 0')
                zones[target_col] = 0.0
            else:
                if len(available) < len(source):
                    print(f'WARNING: some columns missing for "{target_col}": {set(source)-set(available)}')
                zones[target_col] = zones[available].fillna(0).sum(axis=1)
        else:
            if source not in zones.columns:
                print(f'WARNING: source column "{source}" not found for "{target_col}" — column will be 0')
                zones[target_col] = 0.0
            else:
                zones[target_col] = zones[source].fillna(0)

# Verify
present = [c for c in ZONE_ACTIVITY_COLUMNS if c in zones.columns]
missing = [c for c in ZONE_ACTIVITY_COLUMNS if c not in zones.columns]
print(f'Activity columns ready: {present}')
if missing:
    print(f'MISSING (update ZONE_COLUMN_MAP in config.py): {missing}')
else:
    print('All 7 activity columns mapped ✓')
    print(zones[ZONE_ACTIVITY_COLUMNS].describe().round(1))

## 4. Build activity candidate table

In [ ]:
def parse_labels(val):
    if pd.isna(val) or val is None: return []
    if isinstance(val, list): return val
    try: return ast.literal_eval(str(val))
    except: return []

df_final['mid_label_list'] = df_final['mid_label'].apply(parse_labels)

candidates = []
for _, row in df_final.iterrows():
    labels = row['mid_label_list']
    if not labels: continue
    activities = list({MID_LABEL_TO_ACTIVITY[lb] for lb in labels if lb in MID_LABEL_TO_ACTIVITY})
    if not activities: continue
    worker_acts     = [a for a in activities if a == 'Workers']
    non_worker_acts = [a for a in activities if a != 'Workers']
    for act in worker_acts:
        candidates.append({'gml_id': row['gml_id'], 'volume_m3': row['volume_m3'],
                           'activity': act, 'weight': row['potentials'], 'geometry': row.geometry})
    if non_worker_acts:
        w = row['volume_m3'] / len(non_worker_acts)
        for act in non_worker_acts:
            candidates.append({'gml_id': row['gml_id'], 'volume_m3': row['volume_m3'],
                               'activity': act, 'weight': w, 'geometry': row.geometry})

cand_gdf = gpd.GeoDataFrame(candidates, geometry='geometry', crs=df_final.crs)
print(f'Candidate building-activity pairs: {len(cand_gdf):,}')

## 5. Assign buildings to TAZ zones (centroid within zone)

In [ ]:
zones = zones.reset_index(drop=True)
zones['zone_id'] = zones.index

cand_cents = cand_gdf.copy()
cand_cents['geometry'] = cand_gdf.geometry.centroid
cand_with_zone = gpd.sjoin(cand_cents, zones[['zone_id', 'geometry']], how='left', predicate='within')
cand_with_zone = cand_with_zone.drop_duplicates(subset=['gml_id', 'activity'])
print(f'Assigned to zone: {cand_with_zone["zone_id"].notna().sum():,}')
print(f'No zone found:    {cand_with_zone["zone_id"].isna().sum():,}')

## 6. Redistribute zone demand to buildings (local → neighbour → global fallback)

In [ ]:
from shapely.strtree import STRtree
zone_geoms = list(zones.geometry)
zone_tree  = STRtree(zone_geoms)

def get_neighbor_zones(zone_id, n=3):
    geom = zone_geoms[zone_id]
    idxs = zone_tree.nearest(geom, return_distance=False, exclusive=True)
    return list(idxs[:n])

allocation_log = []
assigned_cols  = {act: {} for act in ZONE_ACTIVITY_COLUMNS}

for act in ZONE_ACTIVITY_COLUMNS:
    if act not in zones.columns:
        continue
    for zone_id, zone_row in zones.iterrows():
        demand = zone_row.get(act, 0)
        if pd.isna(demand) or demand <= 0:
            continue
        mask_act  = cand_with_zone['activity'] == act
        mask_zone = cand_with_zone['zone_id']  == zone_id
        pool = cand_with_zone[mask_act & mask_zone].copy()
        if pool.empty:
            pool = cand_with_zone[mask_act & cand_with_zone['zone_id'].isin(get_neighbor_zones(zone_id))].copy()
        if pool.empty:
            pool = cand_with_zone[mask_act].copy()
        if pool.empty or pool['weight'].sum() <= 0:
            continue
        total_w = pool['weight'].sum()
        for _, brow in pool.iterrows():
            share  = brow['weight'] / total_w * demand
            gid    = brow['gml_id']
            assigned_cols[act][gid] = assigned_cols[act].get(gid, 0) + share
            allocation_log.append({'zone_id': zone_id, 'gml_id': gid, 'activity': act, 'assigned': share})

print(f'Allocation log entries: {len(allocation_log):,}')

## 7. Assemble building-level result and save

In [ ]:
result = df_final[['gml_id', 'volume_m3', 'bosserhof_class_norm', 'mid_label', 'potentials', 'geometry']].copy()
for act in ZONE_ACTIVITY_COLUMNS:
    col = f'assigned_{act.replace("-", "_")}'
    result[col] = result['gml_id'].map(assigned_cols.get(act, {})).fillna(0)

result.to_file(REDISTRIBUTION_FILE, driver='GPKG')
pd.DataFrame(allocation_log).to_csv(REDISTRIBUTION_LOG, index=False)
print(f'Saved → {REDISTRIBUTION_FILE}')
print(f'Saved → {REDISTRIBUTION_LOG}')

## 8. Validate: TAZ totals conserved

In [ ]:
log_df = pd.DataFrame(allocation_log)
rows   = []
for act in ZONE_ACTIVITY_COLUMNS:
    if act not in zones.columns: continue
    act_log = log_df[log_df['activity'] == act] if not log_df.empty else pd.DataFrame()
    for zone_id, zone_row in zones.iterrows():
        demand = zone_row.get(act, 0)
        if pd.isna(demand) or demand <= 0: continue
        assigned = act_log[act_log['zone_id'] == zone_id]['assigned'].sum() if not act_log.empty else 0
        rows.append({'zone_id': zone_id, 'activity': act, 'demand': demand,
                     'assigned': assigned, 'abs_diff': abs(demand - assigned)})

val_df = pd.DataFrame(rows)
if not val_df.empty:
    print(f'Max absolute difference (should be ~0): {val_df["abs_diff"].max():.6e}')
    val_df.to_csv(REDISTRIBUTION_VALIDATION, index=False)
    print(f'Saved → {REDISTRIBUTION_VALIDATION}')